In [ ]:
#General Dependencies
from pathlib import Path
import shutil, glob, re
import pandas as pd
import numpy as np
from natsort import natsorted
from file_read_backwards import FileReadBackwards
# from itertools import islice

#RDKit Dependencies
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import BondType
from rdkit.Chem import rdMolTransforms
from rdkit.Chem import GetPeriodicTable
from rdkit.Chem import PeriodicTable

"""
### Notebook Overview:
Written by @GCH v1.0 - last updt. 05/05/2026

This notebook contains code to perform more project-specific DFT validation and to extract relevant data from validated ORCA 5.0.3 output (.out) files and write output files (ex: .xyz, .sdf) from the DFT-optimized .out files. Once the DFT.out files are validated, extract dE for arenes/arynes and calculate a dE(dehydration) = dE(Arene) ==> dE(Aryne) + dE(H2) in kcal/mol. Outputs a 'Calculated_Dehydration_Energies.csv' to the /Project_dir and prepares files/directories for Featurization/Parameterization. 

### Planned Features:
1. Ensure all methods are commented/have documentation
2. Better Error-Handling/exceptions
3. Fine-tune the cutoffs for bond breaking/formation checks/isomerization

### Motivation:
The "Bacon" notebook is intended to do a strict pass/fail analysis on raw DFT .out files. Bacon will indicate whether a DFT
.out is valid or not (completed successfully, converged to a min., no imaginary freqs) but not if the calculation itself was
actually successful/chemically meaningful for a specific task. Ex: In the current arynes case, it's possible for an aryne to 
break open during optimization. These calculations can converge successfully but are essentially meaningless in the context
of a dataset curation/ML-training task as the calculation itself is invalid. This notebook is a futher layer of validation on
calculations and also is a means of generating accessory files necessary for archival or dataset curation online (.xyz/.sdfs).

### How to Use this Notebook:
1. Define relevant Paths to necessary files in the PATH cell, below.
2. Define relevant output Paths for new files to be written to in the PATH cell, below.
3. Run the notebook from the top-level project directory to process .outs and generate .xyz and .sd-files
4. There are several project-specific cells in this notebook find and exclude any arynes that broke open during optimization
5. Once validated, DFT energies are extracted from .outs and used to calculate dE(dehyd.)
6. An output .csv containing dE(dehyd.) data is written to a loccal dir
7. The final cell will operate on all specified directories to ensure that only validated .outs/.xyzs/sdfs remain for analysis

### Example:
I have a dir with 8760 putatively valid DFT .out files in /Aryne_Orca_DFT_Outputs/ that I've validated using Bacon (or some
equiv software, ex: AQME QCORR). I want to generate an .xyz file containing the optimized molecular geometry for each valid
calculation. I also want to generate an .sd-file (.sdf) for long-term archival, as .sdfs contain connectivity explicitly. I 
run all the cells in this notebook and discover that 66 of my arynes broke open during optimization. Thus at the end of this
process I should have 8694 final .outs, .xyzs, and .sdfs in their respective top level dirs within my .outs dir.

### Program Details
Input: Directories containing Orca 5.0.3 DFT output files (opt+freq; M06-2X-D3 / def2-SVP)

### Validation/GeometryExtraction/File Generation ###
-Extracts optimized .xyz coordinates from Orca 5.0.3 DFT output files and outputs .xyz files to /optimized_xyz_files/
-Uses generated .xyz files to create new .sdf files containing the optimized geometry, output to /Optimized_SDFs/

### Find and remove broken/invalid arynes from the dataset
-For aryne structures, compares the pre/post-opt .sdf files for bond cleavage to ID arynes broken open during optimization
-Removes broken arynes to arynes/success/broken_arynes to remove them from the dataset, output to /Excluded_Aryne_Outs/

### Truncate Dataset to include only entries for "complete" reactions (Arene => Aryne + H2)
- Identifies validated but unpaired .out files in the /success/ dirs that don't form dehydrogenation pairs
- Moves those non-utilized arene/aryne.outs to /excluded subdir within each Arene/Aryne subdirs

### Calclulate dE(dehydrogenation) for remaining pairs
- for any remaining structures, calculate the dE(dehydrogenation) between validated structures
- Output a cleaned .csv file with data pertaining only to validated .out/.sdf/.xyz/RDKit mol structures

### Clean local directories for downstream parameterization 
- Removes any unpaired files to subdirs to ensure a 1:1 match for directory parsing during parameterization
"""

In [ ]:
"""
Define Relevant Paths in this cell.
"""

#top level project dir (TLD)
project_dir = Path.cwd().resolve().parent

#working dir for Module6 notebook
module6_dir = project_dir / "Module6_Extract_Dehydration_Energies"

#dirs containing 'Bacon'-processed 'orca.out' DFT output files (or in theory any opt+freq .out files)
arene_dft_output_path = project_dir/"DFT_Arene_Data"/"Arene_Orca_DFT_Outputs"
aryne_dft_output_path = project_dir/"DFT_Aryne_Data"/"Aryne_Orca_DFT_Outputs"

#dirs to contain post-optimization .xyz files (extracted from Orca DFT .outs)
arene_post_opt_xyz_path = arene_dft_output_path/"Arene_Opt_xyz_Files"
aryne_post_opt_xyz_path = aryne_dft_output_path/"Aryne_Opt_xyz_Files"
arene_post_opt_xyz_path.mkdir(parents=True, exist_ok=True)
aryne_post_opt_xyz_path.mkdir(parents=True, exist_ok=True)

#dirs to contain post-optimization .sd-files (generated using DFT .xyz data)
arene_post_opt_sdf_path = arene_dft_output_path/"Arene_Post_Opt_sdfs"
aryne_post_opt_sdf_path = aryne_dft_output_path/"Aryne_Post_Opt_sdfs"
arene_post_opt_sdf_path.mkdir(parents=True, exist_ok=True)
aryne_post_opt_sdf_path.mkdir(parents=True, exist_ok=True)

#dirs which contain the pre-optimization .sdf files (used to generate DFT .inps/fix the bad SDF connect.)
arene_pre_opt_sdf_path = project_dir/"DFT_Arene_Data"/"Arene_Orca_DFT_Inputs"/"Arene_PreOpt_sdfs"
aryne_pre_opt_sdf_path = project_dir/"DFT_Aryne_Data"/"Aryne_Orca_DFT_Inputs"/"Aryne_PreOpt_sdfs"
arene_pre_opt_sdf_path.mkdir(parents=True, exist_ok=True)
aryne_pre_opt_sdf_path.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_filepaths_in_target_dir(target_directory: Path, extension: str, printing=True):
    """
    Function: returns a list of Path objects pertaining to files in a specified directory. Searches for specific .ext
    
    Input:
        - target_directory; Path containing files with the '.extension'
        - extension; str: a string specifying an extension to use, ex: ".out", ".inp", ".sdf" etc.
        - printing; bool; Default value = True; controls printing back to user
        
    Returns: 
        - filepaths - a list of filepaths for each file wit .extension' in 'target_directory'
    """
    
    #grab user extension with a search wildcard
    file_extension = f"*{extension}"
    
    #gather and sort .ext files in /target_directory/
    filepaths = target_directory.glob(file_extension)
    filepaths = natsorted(filepaths, key=str)

    #return some user info
    if printing:
        print(f"\033[1mFound {len(filepaths)} {extension} files in '/{target_directory.stem}/\033[0m'")

    #return the list of located filepaths
    return filepaths

In [ ]:
def write_opt_xyz_files_orca(outputs_dir: Path, destination_dir: Path):
    """
    Function:
        1. Extracts optimized DFT coordinates from Orca 5.x output files contained in '/target_directory/'
        2. If one doesn't exist, writes a formatted .xyz file for each output file to '/destination_dir/'
        
    Input:
        - outputs_dir: Path; a directory containing Orca dft .out files (output files)
        - destination_dir: Path; a directory that will contain the newly written .xyz files
        
    Returns:
        - N/A - no direct return but writes .xyz files to specified directory; output text to terminal. 
    """

    #print a notice to user about what is happening
    print(f"\033[1mWriting new .xyz files from DFT-optimized .out files in '/{outputs_dir.stem}/'....\033[0m")

    #get the list of output files in 'target_directory' to extract xyz coords from
    dft_out_paths = get_filepaths_in_target_dir(outputs_dir, ".out")

    #Next loop over all the gathered .out files; from each one, extract the opt. geom as .xyz
    written_count = 0 #keeping track for printing summary to user
    existing_count = 0
    for dft_out_path in dft_out_paths:
        
        #get the output's filename with no extension
        dft_out_name = dft_out_path.stem

        #use the filename to generate a new .xyz file
        xyz_out_name = dft_out_name + ".xyz"
        
        #store a name for the to-be-made xyz file's path 
        new_xyz_path = destination_dir / xyz_out_name

        #Check if an .xyz file already exists
        if new_xyz_path.is_file(): 
            existing_count += 1
            continue

        #if not, make a new one; pull the opt'd geom from the .out file
        else:
            #extract the optimized .xyz coords using the get_opt_geom() method
            opt_geom = get_opt_geom(dft_out_path)

            #count the number of lines in the geom for .xyz header
            num_atoms_in_xyz = len(opt_geom)

            #generate an empty file to populate with data
            new_xyz_path.touch()
            
            #write the .xyz data into the .xyz file
            with open(new_xyz_path, 'w') as file_object:

                #for progress monitoring
                print(f"\tOptimized .xyz file written: '{new_xyz_path.name}'")
                
                #write the number of atoms at the top plus blank line (following .xyz convention)
                file_object.write(str(num_atoms_in_xyz) + "\n\n")
                
                #write the opt geom DF as a string, below
                file_object.write(opt_geom.to_string(header=False, index=False))
                
                written_count += 1 #count that an .xyz was written

    #give user some diagnostics on how many files existed
    if existing_count >= 1:
        print(f"\033[1m\nExisting .xyz files in '/{destination_dir.name}/': {existing_count}\033[0m")

    #give user some diagnostics on how many files were newly made
    if written_count != 0:
        print(f"\033[1m\nOptimized .xyz files written to '/{destination_dir.name}/': {written_count}\033[0m")    

In [ ]:
def sdf_to_mol_rdkit(path_to_sd_file: Path):
    """
    Function: Converts an SD-file containing a single molecule to an RDKit mol object. Returns the mol object. 
    
    Input:
        - path_to_sd_file: Path; a path to a single named .sd-file
        
    Returns:
        - mol: an RDKit MOL object (assuming a single mol in each .sdf)
    """

    #SDMolSupplier is a class that supplies mol data from an SD-file. SDFs can contain many mols in seq
    #so you need to "supply" them one at a time from the sdf. 
    supplier = Chem.SDMolSupplier(path_to_sd_file, removeHs=False, sanitize=True)

    mol = None # assign a default none to mol 
    
    for m in supplier: #for each of the objects pulled from the sdf and retained in "supplier"
        if m is not None: #If RDKit can interpret the object as a mol, 
            mol = m #assign the mol object to "mol"

        else: #if RDkit can't interpret the supplied object,
            
            continue #go to the next object in supplier; N/A here

    #return the mol if it's good
    return mol 

In [ ]:
def sdfs_to_mols_rdkit(path_to_sdfs: Path):
    """
    Function: Given a directory containing many sd-files, get all the .sdfs and convert them to RDKit
        mol objects. Returns a list of RDKit mol objects. 

        1. If there are failed sdf => mol conversions, prints erroneous file names
        2. Creates an /sdfs_not_convertible_to_mols/ dir within /path_to_sdfs/
        3. Moves failed_sdfs (.sdf files not convertible to mols) to dir in #2
        
    Input:
        - path_to_sdfs: Path; a path to directory containing multiple .sd-files
    
    Returns:
        - rdkit_mols: List; a list of RDKit MOL objects read from .sdfs
    """
    
    #get the list of .sdf files in 'target_directory' 
    sdf_paths = get_filepaths_in_target_dir(path_to_sdfs, ".sdf", printing=False)

    #empty lists to keep track of files
    rdkit_mols = []
    failed_sdfs = []
    
    #Loop through each sd-file contained in the specified directory
    for sdf_file in sdf_paths:
        #SDMolSupplier is a class that supplies mol data from an SD-file. SDFs can contain many mols in seq
        #so you need to "supply" them one at a time from the sdf. 
        supplier = Chem.SDMolSupplier(sdf_file)
        #Chem.SDMolSupplier

        for mol in supplier:
            if mol is None:
                print(f'Error Mol: {sdf_file.stem}')
                failed_sdfs.append(sdf_file)
                continue
                
            elif mol is not None:
                print(f"Validated RDKit mol: '{sdf_file.name}'")
                rdkit_mols.append(mol)

    #if there are no errors, code continues past this check        
    if len(failed_sdfs) != 0:
        print(f'\nFailed mols: {len(failed_sdfs)}') #if there ARE errors, print them out, here
        #move those failed .sdf files to an 'error_sdfs" directory
        source_dir = path_to_sdfs
        dest_dir = path_to_sdfs/"sdfs_not_convertible_to_mols"
        dest_dir.mkdir(parents=True, exist_ok=True)

        for failed_sdf in failed_sdfs:
            if failed_sdf.is_file():
                new_file_path = dest_dir/failed_sdf.name
                shutil.move(failed_sdf, new_file_path)
                print(f"Moved '{failed_sdf.name}' to /error_sdfs dir")
                  
    return rdkit_mols 

In [ ]:
def get_opt_geom(dft_output_file_path: Path):
    """
    Function: Parses a DFT .out file (Orca 5.0.3) for the optimized geometry in .xyz coordinates.
        Returns the geometry as a Pandas DF with column format: AtomID X Y Z
    
    Input:
        - dft_output_file_path: Path; A filepath to a DFT output file (Orca)
        
    Returns: 
        - geom_df: a Pandas DF containing the optimized geometry extracted from a DFT .out
    """

    file_name = dft_output_file_path

    captured_geom_blocks = []

    #empty string to store captured geom lines
    geom_block = []
    geom_block_list = []
    
    #regex match pattern
    END_PATTERN = '^-'
    
    #turn off printing as default
    printing = False
    found_start = False

    #read through a file line-by-line
    with open(file_name, "r") as file:
        for line in file:
            #Read through lines until hitting the Final Geometry Block indicator in text
            if line.strip().startswith("CARTESIAN COORDINATES (ANGSTROEM)"):
                #once the line has been found, indicate that it's time to start printing lines
                found_start = True
                #stops the current loop iteration and goes to the next line
                continue
    
            #once a loop iter hits the end pattern, do one of two things:	
            elif re.match(END_PATTERN, line):
    
                #if a second end (the last '---' is hit, stop printing. 
                if printing:
                    found_start = False
                    printing = False
                    #get the current geom block and save it to a list
                    geom_block_list.append(geom_block)
                    #empty the geom block so the next can be captured
                    geom_block = []
                    continue
    
                #if a start has been found, capture lines
                #this skips the first --- right under the start line
                if found_start:
                    printing = True
                continue
    
            #if match is found, printing is "on"
            #print/capture the line
            if printing:
                geom_block.append(line.strip())
    
    #the last geometry in this list should be the optimized one
    #can confirm looking for "*** FINAL ENERGY EVALUATION AT THE STATIONARY POINT ***" line
    optimized_geom_raw = geom_block_list[-1]
    final_geom = []
    for line in optimized_geom_raw[:-1]:
        stripped = line.strip()
        geom_line = stripped.split()
        final_geom.append(geom_line)
        
    #convert the geometry to a pd DF
    geom_df = pd.DataFrame(final_geom, columns=["AtomID", "x", "y", "z"])
    #set the index to be from 1 rather than 0 - we will work from atom index indices
    #geom_df.index = pd.RangeIndex(1, len(geom_df.index) + 1)
    
    #note the atom indices are 0, not 1
    return geom_df

In [ ]:
def compare_mol_connectivity(mol_in: Chem.Mol, atomic_symbols: list[str], xyz_coords: np.ndarray) -> list[str]:
    """
    Function: Distance-based check for bonds that have broken (or formed, in this instance maybe not relevant)

    For each bonded pair in the input, flag if the optimized distance has grown beyond sum_cov_radii +
    breaking_tolerance during DFT geometry optimization. For each non-bonded pair of heavy atoms (Non-H) that 
    are close, flag possible bond formation (currently this is not performed but can uncomment to do it)

    Input:
        - mol_in: Chem.Mol; RDKit Mol object read in from a pre-optimization .sd-file; serves as reference struc.
        - atomic_symbols: list[str]; a list of atomic symbols read in from an optimized .xyz file
        - xyz_coords: np.ndarray; a Numpy array containing DFT-optimized 3D (xyz) coordinates

    Returns: 
        - issues: list[str]; a list of issues found for each mol. 
    """

    ###May need to adjust these threshold values for including/excluding mols
    #In angstroms, a tunable distance cutoff to determine if a bond is breaking/has broken
    #This is how much the distance between a pair of "bonded" atoms can expand
    #before we consider the bond to have broken.  
    breaking_tolerance = 0.5

    #in Angstroms, a tunable distance cutoff to determine if a bond is possibly forming/formed
    #This is how much the distance between a pair of "non-bonded" atoms can shrink
    #before we consider it to be "putatively bonded." 
    forming_tolerance = 0.3

    #not sure what this does yet
    DEFAULT_RADIUS = 0.80

    #in Angstroms, the approx. VDW radii for common organic atoms
    #These are used as a rough ball-and-stick model estimate of the boundaries of interacting atoms
    COVALENT_RADII = {
        "H": 0.31, "C": 0.76, "N": 0.71, "O": 0.66, "F": 0.57, "P": 1.07, "S": 1.05, "Cl": 1.02,
        "Br": 1.20, "I": 1.39, "B": 0.84, "Si": 1.11, "Se": 1.20,
        }

    #init a list to store any found issues
    issues = []

    bonds_in_input_mol = {frozenset((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
                 for bond in mol_in.GetBonds()}

    ### Check bond-by-bond for bonds that have stretched/maybe broken during optimization ### 
    #loop over each pair of atoms that are inferred to be putatively bonded in the input sdf
    for atom_pair in bonds_in_input_mol: 
        a, b = sorted(atom_pair) #assign atoms a and b from each atom pair
        
        a_b_distance_after_opt = float(np.linalg.norm(xyz_coords[a] - xyz_coords[b]))
        #print(f"Atomic Pair distance: {a_b_distance_after_opt}")
        
        # dist = _pair_distance(coords_out, a, b) #calculate the euclidean distance between 
        #r_sum = _cov_radius_sum(atomic_symbols, a, b)
        sum_of_vdw_radii = (COVALENT_RADII.get(atomic_symbols[a], DEFAULT_RADIUS) + COVALENT_RADII.get(atomic_symbols[b], DEFAULT_RADIUS))
        #print(f"Sum of VDW Radii: {sum_of_vdw_radii}")

        #If the post-opt AB distance is longer than 1.5 Angstrongs + the VDW Radii, 
        #Add a "Bond likely Broken" issue to the list of potential issues 
        if a_b_distance_after_opt > sum_of_vdw_radii + 1.0 + breaking_tolerance: # May need to tune the 1.0 val. or breaking_tol. 
            issues.append(
                f"Bond STRETCH WARNING: atoms {a}–{b} "
                f"({atomic_symbols[a]}–{atomic_symbols[b]})"
                f"dist={a_b_distance_after_opt:.2f} Å (sum cov radii={sum_of_vdw_radii:.2f} Å)"
            )

    ## Currently I'm not sure that 'forming bonds' needs to be checked; breaking is much more common 
    ### Next check if any new bonds formed between Non-H atoms during optimization ###
    #Get a list of Heavy Atoms in the optimized geom (non-Hydrogen atoms) 
    heavy_atoms = [i for i, sym in enumerate(atomic_symbols) if sym != "H"]
    num_heavy_atoms = len(heavy_atoms) #how many non-H atoms in the geom

    ### Next check if any new bonds formed between Non-H atoms during optimization ###
    #loop over each of the heavy atoms in the 3D coordinates
    for i in range(num_heavy_atoms):
        
        #make unique pairs of heavy atoms
        for j in range(i + 1, num_heavy_atoms):
            a, b = heavy_atoms[i], heavy_atoms[j]
            
            #check if a and b are already a bond in the input mol
            if frozenset((a, b)) in bonds_in_input_mol:
                continue #if they are already considered bonded, skip them

            #calculate the Eucl. distance between A and B in angstroms
            a_b_distance_after_opt = float(np.linalg.norm(xyz_coords[a] - xyz_coords[b]))

            #Sum the VDW radii of A and B - this is an estimate of the expected bond length 
            sum_of_vdw_radii = (COVALENT_RADII.get(atomic_symbols[a], DEFAULT_RADIUS) + COVALENT_RADII.get(atomic_symbols[b], DEFAULT_RADIUS))

            #If the distance has shrunk below our threshold, consider the heavy atoms bonded/raise a flag
            if a_b_distance_after_opt < sum_of_vdw_radii + forming_tolerance:
                issues.append(
                    f"New CLOSE CONTACT (possible bond formation): "
                    f"atoms {a}–{b} ({atomic_symbols[a]}–{atomic_symbols[b]}) "
                    f"dist={a_b_distance_after_opt:.2f} Å"
                )

    if issues:
        return issues

In [ ]:
def build_mol_from_coords(pre_opt_mol: Chem.Mol, opt_atomic_symbols: list[str], xyz_coords: np.ndarray) -> Chem.Mol:
    """
    Function: Given a pre-optimization SD-File, Generates a new/updated RDkit mol object from optimized 3D (xyz) coordinates. 

    Inputs: 
        - pre_opt_mol: Chem.Mol; an RDkit Mol object imported from a pre-optimization .sd-file
        - opt_atomic_symbols: list[str]; a list of strings corresponding to atomic symbols; extracted from .xyz file
        - xyz_coords: np.ndarray; a Numpy array consisting of optimized 3D coordinates; extracted from .xyz file

    Returns:
        - updated_mol: Chem.Mol; an RDkit Mol object with DFT-optimizec coordinates
    
    Create an RDKit Mol with the ORCA-optimized coordinates, using the
    template's connectivity (bond graph) and atom ordering.

    The template and ORCA output must have the same atom ordering.
    """

    #check that the number of atoms in the .sdf and the .xyz match
    #(should always be true unless you do something loco)
    if pre_opt_mol.GetNumAtoms() != len(opt_atomic_symbols):
        raise ValueError(
            f"Atom count mismatch: SDF has {pre_opt_mol.GetNumAtoms()} atoms, "
            f"ORCA output has {len(opt_atomic_symbols)} atoms."
        )

    # Verify element symbols match
    for i, (sdf_atom, xyz_symbol) in enumerate(zip(pre_opt_mol.GetAtoms(), opt_atomic_symbols)):
        sdf_symbol = sdf_atom.GetSymbol()

        #explicitly check that both lists match in content/order
        if sdf_symbol.upper() != xyz_symbol.upper():
            raise ValueError(
                f"Atom {i}: SDF has '{sdf_symbol}', ORCA has '{xyz_symbol}'."
                "Atom ordering must be identical."
            )

    # Build editable copy with new coordinates
    editable_mol = Chem.RWMol(Chem.Mol(pre_opt_mol))
    conformer = editable_mol.GetConformer()
    for i, (x, y, z) in enumerate(xyz_coords):
        conformer.SetAtomPosition(i, (x, y, z))

    updated_mol = editable_mol.GetMol()

    if updated_mol is not None:
        return updated_mol

    #I think here we should just write an output .sdf immediately - no mussing about

In [ ]:
def identify_broken_arynes(path_to_pre_opt_sdfs: Path, path_to_opt_xyzs: Path):
    """
    Function: Given a pre-optimization .sd-file for an aryne, compare its pre-opt connectivity to its post-opt
        geometry (via an optimized .xyz file) looking for changes in connectivity/isomerisim. Changes suggest
        that bond(s) have broken/formed => significant alteration of the connectivity pre/post optimization. 
        Used as a triage method to find arynes that have broken open during geometry optimization. 
        
    Inputs:
        - path_to_pre_opt_sdfs: Path; a filepath to directory containing .sdf files with pre-optimization geometries
        - path_to_opt_xyzs: Path; a filepath to directory containing .xyz files generated from optimized coords. 
        
    Returns:
        - broken_xyz_paths: list[Path]; a list of paths corresponding to .xyz files that broke open during opt. 
    """

    ### Working with SD-Files
    #gather all the .sd-files in pre-opt .sdf directory; sort them numerically
    pre_opt_sdfs = get_filepaths_in_target_dir(path_to_pre_opt_sdfs, ".sdf", printing=False)

    #init a list to store broken filepaths
    broken_xyz_paths = []
    
    #loop over each sdf in the pre-opt .sdf dir
    for sd_file in pre_opt_sdfs:
        if sd_file.is_file(): #confirm the .sdf exists
            
            #Attempt to convert incoming .sdf to RDKit mol
            mol_in = sdf_to_mol_rdkit(sd_file)

            #if mol was successfully imported, get the name of the .sdf for matching an .xyz
            if mol_in is not None: 
                #print(f"\n{sd_file.name}: Valid mol from sdf") #debug text
                sdf_name = sd_file.stem #get the name of the file 
                matched_xyz_name = f"{sdf_name}_conf_1.xyz" #infer the name of the matching .out
                matched_xyz_path = path_to_opt_xyzs / matched_xyz_name #path where the .out should be

                ##Below we extract the optimized .xyz coords from the generated .xyz files
                if matched_xyz_path.is_file(): #confirm the .out file exists
                    #print(f"\tFound matching .xyz: {matched_xyz_path.name}")
                    #want to extract the .xyz coords and atom IDs from this small file instead of an .out
                    try:
                        with open(matched_xyz_path, 'r') as file:
                            #init a list to store the geom lines from an .xyz
                            xyz_lines = []

                            # #start appending .xyz lines from index 2 (3rd line)
                            # for line in islice(file, 2, None):
                            #     stripped = line.strip()
                            #     geom_line = stripped.split()
                            #     xyz_lines.append(geom_line)

                            for i, line in enumerate(file):
                                if i >= 2:  # Start from index 2 (3rd line)
                                    stripped = line.strip()
                                    geom_line = stripped.split()
                                    xyz_lines.append(geom_line)

                        #explicitly close the file to clear mem
                        file.close()
                        
                    #throw an exception if the matching .xyz can't be opened for parsing
                    except Exception as e:
                        print(f"Could not read .out {matched_xyz_path.name}: {e}")     

                #If an optimized .xyz file doesn't exist, skip the .sdf 
                elif not matched_xyz_path.is_file():
                    #print(f"Did not find matching .xyz: {matched_xyz_path.name} Skipped.")
                    continue

                #convert the captured lines to a DF
                opt_geom_df = pd.DataFrame(xyz_lines, columns=["AtomID", "x", "y", "z"])

                #get the AtomID col as a list of atomic symbols
                opt_atomic_symbols = opt_geom_df['AtomID'].to_list()

                #convert the 3D coords from the Pd.df to a numpy array
                float_coords = opt_geom_df[["x", "y", "z"]].astype(float)
                opt_xyz_array = float_coords[["x", "y", "z"]].to_numpy()
                
                ## Now we have a valid mol from an .sdf (and its geometry/connections)
                #we also have a list of atoms from the optimized geometry as "opt_atomic_symbols"
                #and we have a np.array of the optimized .xyz coordinates
                broken_mols = 0 #init a counter
                issues = compare_mol_connectivity(mol_in, opt_atomic_symbols, opt_xyz_array)

                if not issues:
                    print(f"\tValid Molecule: {matched_xyz_path.name}")

                if issues:
                    print(f"Broken Molecule: {matched_xyz_path.name}")
                    broken_xyz_paths.append(matched_xyz_path)

                # #need to do this AFTER checking for connectivity changes; no need to write a mol until ready to output a .sdf
                # #build a new RDKit mol object using the original mol/connectivity as a template
                # #populate it with the optimized xyz coordinates extracted from the matched .xyz
                # mol_out = build_mol_from_coords(mol_in, opt_atomic_symbols, opt_xyz_array)

                # #check that the generated mol with updated/DFT-optimized xyz coords is RDKit-valid
                # if mol_out is not None:
                #     print(f"\tGenerated a valid RDkit Mol with updated xyz coordinates for {matched_xyz_path.name}")

    if broken_xyz_paths:
        return broken_xyz_paths

In [ ]:
def exclude_broken_arynes(broken_aryne_path_list: list[Path]):
    """
    Function: Given a list of filepaths to .xyzs that have broken open, remove the offending
        .xyz and its matching .out to "broken" directories within their parent dirs. 

    Inputs:
        - broken_aryne_path_list: list[Path]; a list of aryne filepaths that have broken as
            identified by the 'identify_broken_arynes()' method. 

    Returns:
        - N/A; operates directly on filepaths/directories
    """

    #make a directory to store broken xyz's 
    broken_xyz_dir = aryne_post_opt_xyz_path / "Broken_Aryne_xyzs"
    broken_xyz_dir.mkdir(parents=True, exist_ok=True)

    #make a directory to store broken .outs (in /Failed_Calculations)
    broken_out_dir = aryne_dft_output_path / "Failed_Calculations" / "Broken_Open_During_Opt"
    broken_out_dir.mkdir(parents=True, exist_ok=True)

    for broken_aryne_xyz_path in broken_aryne_path_list:
        broken_xyz_name = broken_aryne_xyz_path.stem

        #find the corresponding .out in the aryne_outputs dir
        broken_out_name = f"{broken_xyz_name}.out"
        print(broken_out_name)
        broken_out_source = aryne_dft_output_path / broken_out_name

        #If the matching .out exists, try to move it to the /failed dir
        if broken_out_source.is_file():
            print("out found")
            try:
                #write a destination dir
                broken_out_destination = broken_out_dir / broken_out_name 

                broken_out_source.rename(broken_out_destination)
                print(f"Moved '{broken_out_name}' to '/{broken_out_source.name}/'") 
            
            except OSError as e:
                print(f"Error moving '{broken_out_name}': {e}")

            except FileNotFoundError: #no matching .out (should never happen)
                print(f"Error: {broken_out_name} was not found.")

        #with the matching .out moved, now sequester the .xyz file, as well
        broken_xyz_destination = broken_xyz_dir / broken_aryne_xyz_path.name 

        try:
            #try to move the broken .xyz to its own sequestered dir
            broken_aryne_xyz_path.rename(broken_xyz_destination)
            #print(f"Moved '{broken_aryne_xyz_path.name}' to '/{broken_xyz_destination.name}/'") 
        
        except OSError as e:
            print(f"Error moving '{broken_aryne_xyz_path.name}': {e}")

        except FileNotFoundError: #no matching .out (should never happen)
            print(f"Error: {broken_aryne_xyz_path.name} was not found.")

In [ ]:
def write_optimized_sdfs(path_to_out_files: Path, path_to_xyz_files: Path, path_to_pre_opt_sdfs: Path, new_sdf_dir: Path): 
    """
    Function: Write a new valid .sdf containing a DFT-optimized 3D geom (as .xyz) from an existing pre-optimization .sdf
        - Make a copy of old .sdf; Replace pre-opt 3D coords in old .sdf with optimized.xyz geom formatted for .sdf
        - Retain connectivity/charge/etc from original .sdf 
        - avoid using any eternal dep.
        
    Inputs:
        - path_to_out_files: Path; directory containing optimized/validated .out files
        - path_to_xyz_files: Path; directory containing optimized .xyz files extracted from .outs
        - path_to_pre_opt_sdfs: Path; directory containing pre-DFT .sdfs (used to write .inps)
        - new_sdf_dir: Path; directory to write the newly written .sdfs to
        
    Returns:
        - N/A - operates on dirs and files without a return
    """

    #print a notice to user about what is happening
    print(f"\033[1mWriting new .sd-files from DFT-optimized .out files in '/{path_to_out_files.stem}/'....\n\033[0m")

    #string format for a row in the atoms block of an .sdf 
    # {X} {Y} {Z} {AtomID} {massdif} {massdif} {chg} {chg} {chg} ... 16 elements
    sdf_str_format = "{:>10}{:>10}{:>10} {:<3} {}  {}  {}  {}  {}  {}  {}  {}  {}  {}  {}  {}"

    #Specify dirs containing necessary files
    out_files = get_filepaths_in_target_dir(path_to_out_files, ".out") #DFT output files
    xyz_files = get_filepaths_in_target_dir(path_to_xyz_files, ".xyz") #DFT-optimized .xyz's
    pre_sdfs = get_filepaths_in_target_dir(path_to_pre_opt_sdfs, ".sdf") #.sdfs used to gen .inps

    #Debug/Transparency - report the number of files in each spec. dir
    print("Checking for necessary files in /target_dirs/...")
    print(f"Found '{len(out_files)}' validated output.outs in '/{path_to_out_files.stem}/'")
    print(f"Found '{len(xyz_files)}' optimized .xyzs in '/{path_to_xyz_files.stem}/'")
    print(f"Found '{len(pre_sdfs)}' pre-opt.sdfs in '/{path_to_pre_opt_sdfs.stem}/'")

    #make a dir for new .sdfs to be written to
    print(f"Writing new .sdfs to: '/{new_sdf_dir.stem}/'")
    
    #Verify that matching .xyz and pre-opt.sdf files are found for each .out   
    for out_file in out_files:
        #Get the mol_ID header (the first 2 elements of the file name) for matching
        out_file_name = out_file.stem
        split_name = out_file_name.split("_") #splits the filename on underscore char
        out_mol_id = f"{split_name[0]}_{split_name[1]}_*" #format a new str to search for
        print(f"\nWorking file: '{out_file.name}'")
        
        #flags indicating presence of targeted files - start false and True if found
        found_sdf = False
        found_xyz = False
        
        #check the /sdf_dir for a matching mol_ID.sdf
        matched_sdf = path_to_pre_opt_sdfs.glob(out_mol_id) #using our formatted header
        
        if matched_sdf: #If a matched file is found, 
            for match in matched_sdf: #could potentially be multiple IDs? 
                #get the path of the matched file
                matched_sdf_path = match #record the path of the target file
                found_sdf = True #turn flag to True
                print(f"\tFound matching .sdf: '{matched_sdf_path.name}'")
        
        else: #no matching .sdf found 
            print(f"\tFound no matching .sdf for '{out_file.name}'\n")
            break
        
        #check the /xyz_dir for a matching mol_ID.xyz
        matched_xyz = path_to_xyz_files.glob(out_mol_id)
        if matched_xyz:
            for match in matched_xyz:
                matched_xyz_path = match #record the path of the target file
                found_xyz = True
                print(f"\tFound matching .xyz: '{matched_xyz_path.name}'")
        
        else: #no matching .xyz found
            print(f"\tFound no matching .xyz for '{out_file.name}'\n")
            break
    
        #Copy the pre-opt.sdf to a new dir
        if found_sdf and found_xyz: #both matching files were located
            print(f"Copying '{matched_sdf_path.name}' to '/{new_sdf_dir.stem}/'")
            
            #set up a target dir / name of file to copy
            new_sdf_name = f"{out_file_name}_post_opt.sdf"
            new_sdf_path = new_sdf_dir / new_sdf_name
    
            # Check if the destination file already exists before copying
            if not new_sdf_path.is_file():
                shutil.copy(matched_sdf_path, new_sdf_path)
                print(f"\tCopied '{matched_sdf_path.name}' to /{new_sdf_dir.stem}/")

                #Verify that the copied .sdf just contains a single mol (remove extraneous confs)
                mol_footer = '$$$$' #separates mol blocks within an .sdf
                found_footer = False
                truncated_content = [] #will store the first mol only for rewrite
                
                try: #look for instances of the "$$$$" string in the text - separates mols
                    with open(new_sdf_path) as file:
                        for line in file:
                            if mol_footer in line and not found_footer: #if you find footer, this is the end
                                truncated_content.append(line) #save the last line
                                found_footer = True #turn on the marker
                                break #stop after the first match/exit the loop
                                
                            elif not found_footer: #append lines until we find the footer 
                                truncated_content.append(line)
                
                except IOError as e: #kind of a placeholder - already checked that it exists and should open
                    print(f"Error reading file: {e}")
                    return 

                try: #now overwrite the captured lines for the first mol only to the .sdf 
                    with open(new_sdf_path, 'w') as f:
                        f.writelines(truncated_content)
                        #print(f"File '{new_sdf_path.name}' truncated successfully after the first occurrence of '{mol_footer}'.") #debug
                        
                except IOError as e:
                    print(f"Error writing to file: {e}")
                
            else: #if it exists, skip the copy - maybe overwrite anyway is better
                print(f"\t'{matched_sdf_path.name}' already exists in '/{new_sdf_dir.stem}/'. Copy skipped.")

        #get the .xyz geom from the post-opt .xyz dir
        try:
            with open(matched_xyz_path, 'r') as file:
                xyz_lines = file.readlines() #pull in all .xyz lines
                xyz_geom_lines = xyz_lines[2:] #the geom starts at the 3rd line
                
                #make a dict of the .xyz 
                xyz_dict_list = [] 
                xyz_cols = ['atom_ID', 'x', 'y', 'z']
                
                for xyz_line in xyz_geom_lines: #we just want lines that correspond to the geom
                    split_xyz_line = xyz_line.split() #split on whitespace
                    
                    if len(split_xyz_line) == len(xyz_cols): #pull the xyz data into a dict
                        row_dict = dict(zip(xyz_cols, split_xyz_line))
                        xyz_dict_list.append(row_dict)
                        
                    else: #hopefully never happens
                        print(f'Len split xyz line: {split_xyz_line}')
                        print(f"Skipping line due to format mismatch.")

        except FileNotFoundError: #no matching .xyz (should never happen)
            print(f"Error: {matched_xyz_path.name} was not found.")

        #now have a copy of pre-opt.sdf in post-opt-sdf dir
        #We want to keep all of the valence/charge/connectivity info from the OG .sdf since that is "good"
        #in the eyes of RDkit => We will replace only the XYZ coords from the DFT-optimized geom
        try: #here we are finding the block of text we will need to parse/replace (the atom block)
            with open(new_sdf_path) as file:
                lines = file.readlines() #probably OK since sdf in this case is small
                #print(lines)

            if len(lines) >= 4: #the "counts line" starts on the 4th line (3rd index)
                fourth_line = lines[3] #we are searching the 4th line for # of atoms (first element of 4th line)
                match = re.search(r'\d+', fourth_line) #mach it wtih regex for digits

                if match: #if you've found a match, what is the int value? that's the num atoms
                    num_atoms = int(match.group())
                    #print(f'\t{new_sdf_path.name} contains {num_atoms} atoms.') #debug
            
            #Next we want to retain the useful portion of the "atoms block" - atomID, charge, etc. 
            #this block is of length = num of atoms, so get a slice of the lines from the 4th line to the # of atomsth line
            # aka: (num_atoms + 4 preceding lines)
            len_geom = num_atoms + 4
            old_geom_lines = lines[4:len_geom] #get everything up to the #
            
            old_valence_data = [] #stores the atomID, charge/valence/etc. cols of .sdf
            
            #set up to convert this to a df for easier manipul./dropping cols/reordering
            retained_cols = ['atom_ID',
                             'c1', 'c2', 'c3', 'c4',
                            'c5', 'c6', 'c7', 'c8',
                             'c9', 'c10', 'c11', 'c12'
                            ]
            
            #loop over lines in the atom block and remove just the xyz columns (first 3 elements)
            #these data are stored in 'old_valence_data' to be combined with a new set of xyz's
            for line in old_geom_lines:
                split_line = line.split()
                old_valence_data.append(split_line[3:]) #saving the atomID =>
            
            #store as a list of formatted data;
            old_sdf_data_list = [] #will be converted to a df
            
            #convert each row of the geom section to col-matched dict
            for processed_line in old_valence_data: 
                
                # Ensure the processed line has the same number of items as columns
                if len(processed_line) == len(retained_cols):
                    
                    # Create a dictionary for the row and append to the list
                    row_dict = dict(zip(retained_cols, processed_line))
                    old_sdf_data_list.append(row_dict)
                    
                else:
                    print(f"Skipping line due to format mismatch.")
            
            #convert .xyz dict to a DF
            new_xyz_df = pd.DataFrame(xyz_dict_list)
        
            #convert to a DF to append in the .xyz data
            old_sdf_df = pd.DataFrame(old_sdf_data_list)

            #now combining the old SDF atoms block (sans xyz) with the optimized xyz coords
            #check that the atom_ID cols are equal (same atoms/ordering)
            if new_xyz_df['atom_ID'].equals(old_sdf_df['atom_ID']):

                dropped_xyz = new_xyz_df.drop('atom_ID', axis=1)
                dropped_xyz = dropped_xyz.astype(float)
                rounded_xyz = dropped_xyz.round(4)
            
                new_sdf_df = pd.concat([rounded_xyz, old_sdf_df], axis=1)

                formatted_strings = [sdf_str_format.format(row[0], row[1], row[2], row[3],
                                                          row[4], row[5], row[6], row[7],
                                                          row[8], row[9], row[10], row[11],
                                                          row[12], row[13], row[14], row[15]) for row in new_sdf_df.values.tolist()] 

                #the lines to enter have to have explicit newline characters at the end; ensure they do, here
                formatted_replacement = [line if line.endswith('\n') else line + '\n' for line in formatted_strings]
                
                # #test our formatted .sdf atoms block
                # for line in formatted_replacement:
                #     print(line) #(it looks good)

                #now we do the replacement by reassigning our formatted_replacement lines to the old slice:
                lines[4:len_geom] = formatted_replacement
                
            else:
                print("df matching seems messed up")

            #now we have a copied/parsed .sdf and the block of text we want to replace into it
            #we next need to do the actual substitution in the copied .sdf
            with open(new_sdf_path, "w") as file:
                file.writelines(lines)
 
        except FileNotFoundError: #no matching .sdf (should never happen)
            print(f"Error: {new_sdf_path.name} was not found.")

In [ ]:
def get_scf_energy_orca5(path_to_out_file: Path):
    """
    Function: Extract the last SCF energy from an opt'd geom

    Input:
        -path_to_out_file: Path; path to an Orca 5.0.3 DFT .out file 
        
    Returns: 
        - SCF Energy (dE) in Hartrees
    """

    #Orca5 reports SCF energies using this str line format (the energy immediately follows as the 5th element)
    scf_line_marker = "FINAL SINGLE POINT ENERGY"

    #read through the file backwards matching for our desired text line
    if path_to_out_file.is_file():
        try:
            #read the .out file back to front; the first SCF line is the final SCF energy of the calc.
            with FileReadBackwards(path_to_out_file, encoding="utf-8") as out_file:
                for line in out_file: #remember we are going backwards
    
                    #Reading from back to front, the first SCF line is the final energy 
                    if scf_line_marker in line:
    
                        #split the matched line on whitespace delimiter
                        split_line = line.split()
                        
                        #the energy is 5th element; convert it to float
                        final_energy = float(split_line[4])
    
                        #round the val to 4 dps
                        final_energy_rounded = round(final_energy, 4)
    
                        #exit as soon as the first match is found (the last SCF line)
                        break 
                        
                else:  #can't read the energy for some reason
                    print(f'{file_path.name} does not contain a final SCF energy - Check your calculation.')
    
        except Exception as e:
                print(f"Could not read .out {path_to_out_file.stem}: {e}")
            
    elif not path_to_out_file.is_file():
        print(f"\t{path_to_out_file.stem} not found in '/{path_to_out_file.parent.name}/'")
        
    #return just the final energy
    return final_energy_rounded

In [ ]:
def extract_dE_to_dataframe(df_column, target_directory: Path):
    '''
    Function: Given a dataframe containing a column of molecular IDs, for example (Arene_1, ..., Arene_n), look
        in the passed directory for a corresponding Orca DFT .out file. If the file exists in passed dir, attempt
        to extract the final SCF energy (in Hartrees) from the .out file. Save energies to a list; if the energy
        can't be read, append a 'NaN'. Returns the list of parsed energies/NaNs. 

    Input:
        -df_column - a column in a Pandas DF that contains mol ID headers - in this case "arene_x" or "aryne_y"
        -target_directory: Path; a directory containing .out files
        
    Returns:
        - scf_energies: list - a list of floats/NaNs corresponding to extracted SCF energies (hartrees)
    '''

    scf_energies = []

    #Get the mol_ID from the mol_ID col of a DF (ex: arene_3) 
    for mol_id in df_column: 

        #convert the dir passed to a str (constructing a path to the .out file)
        str_dir = str(target_directory)

        #build the name of the formatted .out file using the captured mol_ID
        out_name = Path(mol_id+"_rdkit_conf_1.out")

        #Define the full path to the .out file 
        log_file = str_dir/out_name
        
        #if the file exists, attempt to extract the final SCF energy
        try:
            energy = get_scf_energy_orca5(log_file) 
            
        #If unreadable (.out doesn't exist or error in calc), append a NaN  
        except:
            energy = np.nan #append NaN for any unreadable energies

        #append the energy (either # or NaN to a list)
        scf_energies.append(energy)

    #return the list of scf_energies
    return scf_energies

In [ ]:
""" Extract Geometries and Generate Optimized .XYZ Files for ARENES 

This cell will generate .xyz files using validated .out files. The optimized geometry obtained via DFT
will be written to the target files. Files are written to a specified output dir. 
"""

#Generate optimized.xyz files from successful Arene Orca DFT calcs
write_opt_xyz_files_orca(arene_dft_output_path, arene_post_opt_xyz_path)

In [ ]:
""" Extract Geometries and Generate Optimized .XYZ Files for ARYNES 

This cell will generate .xyz files using validated .out files. The optimized geometry obtained via DFT
will be written to the target files. Files are written to a specified output dir. 
"""

#Generate optimized.xyz files from successful Aryne Orca DFT calcs
write_opt_xyz_files_orca(aryne_dft_output_path, aryne_post_opt_xyz_path)

In [ ]:
""" ### Identify and Isolate Broken Aryne Calculations ###

This is a project-specific cell used to identify broken arynes (rings that open during optimization).
By comparing pre- and post-opt # of bonds, we can reliably ID arynes that have broken open during optimization. 

The "exclude_broken_arynes" method excludes aryne.sdfs on the basis of a dramatic reduction in # of bonds. 
Arynes that break exhibit fewer net bonds than their pre-opt num of bonds. If post < pre, aryne broke. 
"""

#Print some text to the user
print(f"Parsing Input/Output files for Arynes that have broken open during optimization...\n")

#check for existing processed (opened) files in the /failed_dir
broken_arynes_dir = aryne_dft_output_path / "Failed_Calculations" / "Broken_Open_During_Opt"

#f this directory exists and has files in it, this cell has been run
if broken_arynes_dir.is_dir() and any(broken_arynes_dir.iterdir()):
    print(f"Found existing broken arynes in '/{broken_arynes_dir.parent.name}/'")

#otherwise, find the broken arynes
else:
    # #compare geometries before/after optimization and search for bonds that have broken
    broken_arynes = identify_broken_arynes(aryne_pre_opt_sdf_path, aryne_post_opt_xyz_path)
    
    #send the broken .outs/.xyzs to the abyss
    if broken_arynes:
        #give some feedback on the number of broken structures
        print(f"\nNumber of arynes identified as broken: {len(broken_arynes)}")
        exclude_broken_arynes(broken_arynes)

In [ ]:
""" ### Write Final Arene.SDFs ###

This cell generates finalized .sd-files (.sdf) for archival.
The atomic connectivity is retained from pre-opt sd-files. 
The geometry is extracted from the DFT calculation (it is a DFT-optimized geom). 
"""

#print cell header
print(f"Writing final .sdfs from optimized .xyz geometries...\n")

#find if there are existing sd-files in the target dir
present_sdfs = get_filepaths_in_target_dir(arene_post_opt_sdf_path, ".sdf")

#if present, we don't remake them
if not present_sdfs:
    #Fields: (.output dir, .xyz dir, pre-opt SDF dir, target dir to write to) 
    write_optimized_sdfs(arene_dft_output_path, arene_post_opt_xyz_path, arene_pre_opt_sdf_path, arene_post_opt_sdf_path)

In [ ]:
""" ### Write Final Aryne.SDFs for archival ###

This cell generates finalized .sd-files (.sdf) for archival.
The connectivity is retained from pre-opt sd-files. 
The geometry is extracted from the DFT calculation (it is a DFT-optimized geom). 
"""

#print cell header
print(f"Writing final .sdfs from optimized .xyz geometries...\n")

#find if there are existing sd-files in the target dir
present_sdfs = get_filepaths_in_target_dir(aryne_post_opt_sdf_path, ".sdf")

#if present, we don't remake them
if not present_sdfs:
    #Fields: (.output dir, .xyz dir, pre-opt SDF dir, target dir to write to) 
    write_optimized_sdfs(aryne_dft_output_path, aryne_post_opt_xyz_path, aryne_pre_opt_sdf_path, aryne_post_opt_sdf_path)    
    

"""
### Cells above: Validating and Generating Output Files, Directory Management
### Cells below: Extract DFT Energies and Output dE(Dehyd.) Dataframe.csv
"""

In [ ]:
""" ### Read in the generated arene/aryne.csv dataset from TLD ### 

This cell reads in the 'arenes_and_generated_arynes.csv' dataset file in the TLD
    We use this DF to access the arene_id/aryne_id cols for naming purposes. A
    copy of these data is created via this notebook and appended with DFT-calculated
    dEs and the dehydration energy before being written to an output .csv. 
    
"""

#The Module3 directory contains the .csv we need in this notebook
arenes_and_arynes_smiles = project_dir / "Module3_Generate_Aryne_SMILES" / "Arenes_and_Generated_Arynes.csv"

### Read in the generated arene/aryne.csv dataset from TLD ### 
#Use this as a naming reference against the files in our output dirs
hetaryne_df = pd.read_csv(arenes_and_arynes_smiles)
hetaryne_df.head()

In [ ]:
"""### Extract SCF Energies from DFT-Optimized ARENES ###

This cell will extract the final single point SCF Energy (dE) from validated Orca Arene.outs
in the /arene_dft_outs dir. The ID-paired energies are appended to a new column in a Pandas DF.
'NaN' is appendend if/when an .out is undreadable/the SCF energy can't be found. 
"""

#Check for an existing .csv containing extracted energies in Mod6 dir
arene_energy_csv = Path("Extracted_Arene_Energies.csv")

#If the reference file exists, don't re-extract energetic data 
if arene_energy_csv.exists():
    print(f"Found an existing 'Extracted_Arene_Energies.csv' in local directory.\n\nImported existing energy data:")

    #check if current DF has energy column in it
    if 'arene_energy' not in hetaryne_df.columns:
    
        #read in the energetic data from the local .csv
        arene_energies = pd.read_csv(arene_energy_csv)

        #append the data to the working hetaryne_df
        hetaryne_df['arene_energy'] = arene_energies['arene_energy'] 

#if the csv doesn't yet exist, extract the energies
else:
    #Extract energies from processed/valid ARENE Orca .out files
    print("Processing Arene Energies...")

    #add the extracted energies as a column to the working df
    hetaryne_df['arene_energy'] = extract_dE_to_dataframe(hetaryne_df['arene_ID'], arene_dft_output_path)

    #write the col to a .csv so we don't have to recalc if re-running notebook
    hetaryne_df[['arene_ID', 'arene_energy']].to_csv('Extracted_Arene_Energies.csv', index=False)

#print the first 5 rows of the DF 
hetaryne_df.head()

In [ ]:
"""### Extract SCF Energies from DFT-Optimized ARYNES ###

This cell will extract the final single point SCF Energy (dE) from validated Orca Aryne.outs
    in the /aryne_dft_outs dir. The ID-paired energies are appended to a new column in a Pandas DF.
    'NaN' is appendend if/when an .out is undreadable/the SCF energy can't be found. 
"""

#Check for an existing .csv containing extracted energies in Mod6 dir
aryne_energy_csv = Path("Extracted_Aryne_Energies.csv")

#If the reference file exists, don't re-extract energetic data 
if aryne_energy_csv.exists():
    print(f"Found an existing 'Extracted_Aryne_Energies.csv' in local directory.\n\nImported existing energy data:")

    #check if current DF has energy column in it
    if 'aryne_energy' not in hetaryne_df.columns:
    
        #read in the energetic data from the local .csv
        aryne_energies = pd.read_csv(aryne_energy_csv)

        #append the data to the working hetaryne_df
        hetaryne_df['aryne_energy'] = aryne_energies['aryne_energy'] 
    
#if the csv doesn't yet exist, extract the energies
else:
    #Extract energies from processed/valid ARENE Orca .out files
    print("Processing Aryne Energies...")

    #add the extracted energies as a column to the working df
    hetaryne_df['aryne_energy'] = extract_dE_to_dataframe(hetaryne_df['aryne_ID'], aryne_dft_output_path)

    #write the col to a .csv so we don't have to recalc if re-running notebook
    hetaryne_df[['aryne_ID', 'aryne_energy']].to_csv('Extracted_Aryne_Energies.csv', index=False)

#print the first 5 rows of the DF 
hetaryne_df.head()

In [ ]:
""" ### Compute dE(Dehydration) Using Arene ==> Aryne + H2 ###

This cell is used to calculate the DFT-derived dehydration energies for the hypothetical reaction 
    "Arene ==> Aryne + H2." The value is calculated from the values within Pandas DF "hetaryne_df".
    The SCF energy (dE) of H2 was calculated at the same LOT and manually entered, below. Values 
    are calculated in Hartree and then converted to Kcal/mol using the appl. conversion factor.
"""

#Compute the dehydrogenation energy given energies present in both arene/aryne columns
## the energy of H2 (same level of theory is -1.16498994) - this comes from the H2 orca calc in top-level dir

#conversion factor between Hartrees and kcal/mol
Hartree_to_kcal = 627.509

#calculate the dehydration energy using Arene ==> Aryne + H2; Round to 4dps; units are kcal/mol
if "dehydrogenation_energy" in hetaryne_df.columns:
    print(f"Found extant 'dehydrogenation_energy' data in current dataframe.\n")
else:
    print(f"Calculating dE(dehydrogenation)...")
    hetaryne_df['dehydrogenation_energy'] = round(((hetaryne_df['arene_energy'] -
                                                    (hetaryne_df['aryne_energy'] + -1.16372587))
                                                   * -1 * Hartree_to_kcal), 4)

hetaryne_df.head()

In [ ]:
""" ### Format and Write data to .csv - Output 'Calculated_Dehydrogenation_Energies.csv' to /ProjectDir ###

This cell reformats the current dehydrogenation dataframe to a preferred column order and sorts ascending
    based on the "aryne_id" column (can also change that, here). Writes a .csv file to the /projectdir 
    called 'Calculated_Dehydrogenation_Energies.csv' containing DFT-calculated dE(dehyd.). 
"""

### Format Output.csv; Remove unitilized arenes/arynes; Output the final dehydrogenation_energies.csv  ###
new_col_order = ['arene_ID', 'arene_smiles', 'arene_energy',
                 'aryne_ID', 'aryne_smiles', 'aryne_energy',
                 'dehydrogenation_energy']

#copy df to change the column ordering
hydrog_df = hetaryne_df[new_col_order]

#Sort the df rows based on the aryne_id# (ascending)
hydrog_sorted = hydrog_df.sort_values(by='aryne_ID', key=lambda col: col.apply(lambda x: int(re.search(r'\d+', x).group())))

#Drop rows that contain a 'NaN' dehydrogenation energy (these don't have all the DFT calcs needed to calc dE)
dehydrogenation_energies_df = hydrog_sorted.dropna(how='any',axis=0)

#output the current DF as a .csv to TLD (project dir)
output_csv = project_dir / "Module6_Extract_Dehydration_Energies" / "Calculated_Dehydrogenation_Energies.csv"
dehydrogenation_energies_df.to_csv(output_csv, index=False)

#print summary to console
print(f"'{output_csv.name}' containing {len(dehydrogenation_energies_df)} DFT-calculated "
    "dehydrogenation energies written to '/{output_csv.parent.name}/'.\n")

#also write a backup .csv to /csv_backups
backup_energy_dir = project_dir / "csv_backups" / "Calculated_Dehydrogenation_Energies"
backup_energy_dir.mkdir(parents=True, exist_ok=True)
backup_energy_csv = backup_energy_dir / "Calculated_Dehydrogenation_Energies.csv"
dehydrogenation_energies_df.to_csv(backup_energy_csv, index=False)

#for debug
dehydrogenation_energies_df.head()

In [ ]:
"""### Final ARENE File Cleanup / Dir management before Parameter Extraction ###

This cell uses the generated "Calculated_Dehydrogenation_Energies.csv" to determine which .outs in 
/Arene_Orca_DFT_Outputs are paired with a "parent" arene. Unpaired arene.outs (those lacking a parent arene)
are moved to a "Valid but unused dir". Given some unused .outs, isolate the corresponding .sdfs and .xyzs 
for unpaired arenes. This ensures a 1:1 match in the number of validated files across all project subdirs.
"""

#Make dirs to hold unpaired files - these are either an arene/aryne missing its partner
#Dir for unpaired Arene.outs
unpaired_arene_outs = arene_dft_output_path/"Valid_but_Unpaired_arenes"
unpaired_arene_outs.mkdir(parents=True, exist_ok=True)

#Dir for unpaired arene_post_opt.sdfs
unpaired_arene_sdfs = arene_post_opt_sdf_path/"Unpaired_Arene_sdfs"
unpaired_arene_sdfs.mkdir(parents=True, exist_ok=True)

#Dir for unpaired arene.xyzs
unpaired_arene_xyzs = arene_post_opt_xyz_path/"Unpaired_Arene_xyzs"
unpaired_arene_xyzs.mkdir(parents=True, exist_ok=True)

#Pull in the dehydrogenation DF as a naming/ID/energy reference
#arenes in this DF are confirmed to be paired; can process anything knowing it is paired
dehyd_df = pd.read_csv('Calculated_Dehydrogenation_Energies.csv')
dehyd_df.head()

#pull IDs of paired 'arene_ids' from dehydrog DF
utilized_arene_ids = dehyd_df['arene_ID'].tolist()

#Need a list of filepaths that ARE used - not all .outs are actually used - need to be paired
#loop through the arene_IDs and generate filepaths to search for
utilized_arene_paths = [] #list to store utilized filepaths
for utilized_arene_id in utilized_arene_ids:
    #Construct the name out the expected .out 
    name_of_out = f"{utilized_arene_id}_rdkit_conf_1.out"
    #dir that the source file should be in
    source_out_path = arene_dft_output_path / name_of_out
    #append the filepath to a list of filepaths
    utilized_arene_paths.append(source_out_path)

#Now, compare the list of utilized arenes against the list of .outs currently in the .outs dir
#Get a list of existing arene filepaths for .out files currently in /arene_dft_output_path
current_arene_outs = get_filepaths_in_target_dir(arene_dft_output_path, ".out")

#Next calculate the set of .outs currently in /arene_dft_output_path that are not utilized
non_utilized_arenes = set(current_arene_outs) - set(utilized_arene_paths)

#Using the list of unutilized IDs, move the unused .outs, .xyzs, and .sdfs to respective /unused dirs
for unutilized_out in non_utilized_arenes:
    #specify the source/destination paths for unutilized .outs
    source_out = unutilized_out
    unutilized_out_name = unutilized_out.name
    destination_out = unpaired_arene_outs / unutilized_out_name

    #attempt to move the .out 
    if source_out.is_file():
        try:
            source_out.rename(destination_out)
            #print(f"Moved '{source_out.name}' to '/{unpaired_arene_outs.name}/'") 
            
        except OSError as e:
            print(f"Error moving '{source_out}': {e}")
            
    else:
        print(f"File '{source_out.name}' not found in '/{source_out.parent}/'")

    #separate the associated unused arene.xyz file
    raw_name = unutilized_out.stem
    xyz_name = f"{raw_name}.xyz"
    xyz_source = arene_post_opt_xyz_path / xyz_name
    xyz_dest = unpaired_arene_xyzs / xyz_name

    #attempt to move the .xyz
    if xyz_source.is_file():
        try:
            xyz_source.rename(xyz_dest)
            #print(f"Moved '{xyz_source.name}' to '/{unpaired_arene_xyzs.name}/'") 
            
        except OSError as e:
            print(f"Error moving '{xyz_source}': {e}")
            
    else:
        print(f"File '{xyz_source.name}' not found in '/{xyz_source.parent}/'")
    
    #separate the associated unused arene.sd-file
    raw_split = raw_name.split("_")
    mol_id = f"{raw_split[0]}_{raw_split[1]}"
    post_sdf_name = f"{mol_id}_rdkit_conf_1_post_opt.sdf"
    sdf_source = arene_post_opt_sdf_path / post_sdf_name
    sdf_dest = unpaired_arene_sdfs / post_sdf_name

    #attempt to move the .sdf
    if sdf_source.is_file():
        try:
            sdf_source.rename(sdf_dest)
            #print(f"Moved '{sdf_source.name}' to '/{unpaired_arene_sdfs.name}/'") 
            
        except OSError as e:
            print(f"Error moving '{sdf_source}': {e}")
            
    else:
        print(f"File '{sdf_source.name}' not found in '{sdf_source.parent}'")    

In [ ]:
"""### Final ARYNE File Cleanup / Dir management before Parameter Extraction ###

This cell uses the generated "Calculated_Dehydrogenation_Energies.csv" to determine which .outs in 
/Aryne_Orca_DFT_Outputs are paired with a "parent" arene. Unpaired aryne.outs (those lacking a parent arene)
are moved to a "Valid but unused dir". Given some unused .outs, isolate the corresponding .sdfs and .xyzs 
for unpaired arynes. This ensures a 1:1 match in the number of validated files across all project subdirs.
"""

#Make dirs to hold unpaired files - these are either an arene/aryne missing its partner
#Dir for unpaired Aryne.outs
unpaired_aryne_outs = aryne_dft_output_path/"Valid_but_Unpaired_Arynes"
unpaired_aryne_outs.mkdir(parents=True, exist_ok=True)

#Dir for unpaired Aryne_post_opt.sdfs
unpaired_aryne_sdfs = aryne_post_opt_sdf_path/"Unpaired_Aryne_sdfs"
unpaired_aryne_sdfs.mkdir(parents=True, exist_ok=True)

#Dir for unpaired Aryne.xyzs
unpaired_aryne_xyzs = aryne_post_opt_xyz_path/"Unpaired_Aryne_xyzs"
unpaired_aryne_xyzs.mkdir(parents=True, exist_ok=True)

#Pull in the dehydrogenation DF as a naming/ID/energy reference
#Arynes in this DF are confirmed to be paired; can process anything knowing it is paired
dehyd_df = pd.read_csv('Calculated_Dehydrogenation_Energies.csv')
dehyd_df.head()

#pull IDs of paired 'aryne_ids' from dehydrog DF
utilized_aryne_ids = dehyd_df['aryne_ID'].tolist()

#Need a list of filepaths that ARE used - not all .outs are actually used - need to be paired
#loop through the aryne_IDs and generate filepaths to search for
utilized_aryne_paths = [] #list to store utilized filepaths
for utilized_aryne_id in utilized_aryne_ids:
    #Construct the name out the expected .out 
    name_of_out = f"{utilized_aryne_id}_rdkit_conf_1.out"
    #dir that the source file should be in
    source_out_path = aryne_dft_output_path / name_of_out
    #append the filepath to a list of filepaths
    utilized_aryne_paths.append(source_out_path)

#Now, compare the list of utilized arynes against the list of .outs currently in the .outs dir
#Get a list of existing Aryne filepaths for .out files currently in /aryne_dft_output_path
current_aryne_outs = get_filepaths_in_target_dir(aryne_dft_output_path, ".out")

#Next calculate the set of .outs currently in /aryne_dft_output_path that are not utilized
non_utilized_arynes = set(current_aryne_outs) - set(utilized_aryne_paths)

#Using the list of unutilized IDs, move the unused .outs, .xyzs, and .sdfs to respective /unused dirs
for unutilized_out in non_utilized_arynes:
    #specify the source/destination paths for unutilized .outs
    source_out = unutilized_out
    unutilized_out_name = unutilized_out.name
    destination_out = unpaired_aryne_outs / unutilized_out_name

    #attempt to move the .out 
    if source_out.is_file():
        try:
            source_out.rename(destination_out)
            #print(f"Moved '{source_out.name}' to '/{unpaired_aryne_outs.name}/'") 
            
        except OSError as e:
            print(f"Error moving '{source_out}': {e}")
            
    else:
        print(f"File '{source_out.name}' not found in '/{source_out.parent}/'")

    #separate the associated unused aryne.xyz file
    raw_name = unutilized_out.stem
    xyz_name = f"{raw_name}.xyz"
    xyz_source = aryne_post_opt_xyz_path / xyz_name
    xyz_dest = unpaired_aryne_xyzs / xyz_name

    #attempt to move the .xyz
    if xyz_source.is_file():
        try:
            xyz_source.rename(xyz_dest)
            #print(f"Moved '{xyz_source.name}' to '/{unpaired_aryne_xyzs.name}/'") 
            
        except OSError as e:
            print(f"Error moving '{xyz_source}': {e}")
            
    else:
        print(f"File '{xyz_source.name}' not found in '/{xyz_source.parent}/'")
    
    #separate the associated unused aryne.sd-file
    raw_split = raw_name.split("_")
    mol_id = f"{raw_split[0]}_{raw_split[1]}"
    post_sdf_name = f"{mol_id}_rdkit_conf_1_post_opt.sdf"
    sdf_source = aryne_post_opt_sdf_path / post_sdf_name
    sdf_dest = unpaired_aryne_sdfs / post_sdf_name

    #attempt to move the .sdf
    if sdf_source.is_file():
        try:
            sdf_source.rename(sdf_dest)
            #print(f"Moved '{sdf_source.name}' to '/{unpaired_aryne_sdfs.name}/'") 
            
        except OSError as e:
            print(f"Error moving '{sdf_source}': {e}")
            
    else:
        print(f"File '{sdf_source.name}' not found in '{sdf_source.parent}'")    

In [ ]:
"""### Diagnostics on Each Directory ###
"""
final_arene_outs = get_filepaths_in_target_dir(arene_dft_output_path, ".out", printing=False)
print(f"Number of validated/paired Arene.outs in dataset: {len(final_arene_outs)}")

final_aryne_outs = get_filepaths_in_target_dir(aryne_dft_output_path, ".out", printing=False)
print(f"Number of validated/paired Aryne.outs in dataset: {len(final_aryne_outs)}")

print(f"\nNumber of DFT-calculated dE(dehydration) values in dataset: {len(dehyd_df)}")